# Stage 3-4. MLP Model

이 노트북은 `src/models/mlp.py`에 구현된 `MLP` 클래스를 실습한다.

MLP는 `src/nn/` 모듈을 조립하여 구성된 3층 완전연결 네트워크이다.

```
Linear(784→256) → Sigmoid → Linear(256→128) → Sigmoid → Linear(128→output_dim)
```

| 속성/메서드 | 설명 |
|---|---|
| `MLP(task, seed)` | task별 output_dim으로 네트워크 생성 |
| `forward(x)` | raw logits `(N, output_dim)` 반환 |
| `backward(grad_out)` | `src.nn.losses` gradient를 입력으로 역전파 |
| `params` | 모든 레이어의 파라미터 리스트 |
| `grads` | 모든 레이어의 gradient 리스트 |

**학습 목표**
1. task별로 `output_dim`과 파라미터 수가 달라지는 구조를 확인한다.
2. `forward` → `backward` → 수동 파라미터 업데이트 흐름(1 step)을 실행한다.
3. 수동 epoch 반복으로 loss 하강 곡선을 시각화한다.
4. 3종 task에 대해 각각 MLP를 생성하고 학습 결과를 비교한다.

## 0. 환경 설정

In [ ]:
import sys
sys.path.insert(0, "../..")

import numpy as np
import matplotlib.pyplot as plt

from src.models.mlp import MLP
from src.task import get_task_spec, transform_targets
from src.nn.losses import (
    cross_entropy, cross_entropy_grad,
    binary_cross_entropy, binary_cross_entropy_grad,
    mse, mse_grad,
)
from src.nn.metrics import accuracy, binary_accuracy, r2_score

## 1. MLP 생성 — task별 구조 확인

In [ ]:
for task in ["multiclass", "binary", "regression"]:
    model = MLP(task=task, seed=0)
    spec  = get_task_spec(task)
    total_params = sum(p.size for p in model.params)

    print(f"task={task}")
    print(f"  output_dim   : {model.output_dim}")
    print(f"  params 수     : {len(model.params)} 개 텐서")
    print(f"  파라미터 합계 : {total_params:,} 개")
    for i, p in enumerate(model.params):
        print(f"    params[{i}] : {p.shape}")
    print()

## 2. Forward pass — logits shape 확인

In [ ]:
model_mc = MLP(task="multiclass", seed=42)

rng = np.random.default_rng(0)
x_batch = rng.normal(0, 1, (32, 784)).astype(np.float32)

logits = model_mc.forward(x_batch)

print("=== Forward pass ===")
print(f"입력 x    : {x_batch.shape}")
print(f"logits    : {logits.shape}")
print(f"logits[0] : {logits[0].round(4)}")
print()
print("forward는 raw logits를 반환한다 (softmax 미적용).")
print(f"logits 합계(첫 샘플): {logits[0].sum():.4f}  ← 1이 아님")

## 3. 1 Step: forward → loss → backward → 수동 업데이트

In [ ]:
model = MLP(task="multiclass", seed=42)
lr = 0.01

# 데이터
x = rng.normal(0, 1, (32, 784)).astype(np.float32)
labels = rng.integers(0, 10, 32).astype(np.uint8)
targets = transform_targets(labels, "multiclass")

# Step 1: Forward
logits = model.forward(x)
print(f"[Forward]  logits  shape: {logits.shape}")

# Step 2: Loss
loss_before = cross_entropy(logits, targets)
print(f"[Loss]     before update: {loss_before:.4f}")

# Step 3: Backward
grad_out = cross_entropy_grad(logits, targets)
model.backward(grad_out)
print(f"[Backward] grad_out shape: {grad_out.shape}")

# Step 4: 수동 파라미터 업데이트
for param, grad in zip(model.params, model.grads):
    param -= lr * grad

# Step 5: Loss 재확인
logits_after = model.forward(x)
loss_after = cross_entropy(logits_after, targets)
print(f"[After]    loss after update: {loss_after:.4f}")
print(f"           loss 감소: {loss_before - loss_after:.6f}")

## 4. 수동 epoch 학습 — multiclass

In [ ]:
# 작은 synthetic 데이터로 epoch 반복 학습
N_TRAIN = 512
EPOCHS  = 40
LR      = 0.05

rng_train = np.random.default_rng(1)
X_train  = rng_train.normal(0, 1, (N_TRAIN, 784)).astype(np.float32)
L_train  = rng_train.integers(0, 10, N_TRAIN).astype(np.uint8)
T_train  = transform_targets(L_train, "multiclass")  # (N, 10)

model_train = MLP(task="multiclass", seed=0)

loss_log   = []
metric_log = []

for epoch in range(EPOCHS):
    logits_e = model_train.forward(X_train)
    l = cross_entropy(logits_e, T_train)
    m = accuracy(logits_e, T_train)
    loss_log.append(float(l))
    metric_log.append(float(m))

    grad_e = cross_entropy_grad(logits_e, T_train)
    model_train.backward(grad_e)

    for p, g in zip(model_train.params, model_train.grads):
        p -= LR * g

print(f"초기 loss : {loss_log[0]:.4f} | 최종 loss : {loss_log[-1]:.4f}")
print(f"초기 acc  : {metric_log[0]:.4f} | 최종 acc  : {metric_log[-1]:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle("MLP Manual Training — multiclass (synthetic)", fontsize=13, fontweight="bold")

ax1.plot(loss_log, color="steelblue", linewidth=2)
ax1.set_title("Cross-Entropy Loss")
ax1.set_xlabel("epoch")
ax1.set_ylabel("loss")
ax1.grid(True, alpha=0.3)

ax2.plot(metric_log, color="#E87B4C", linewidth=2)
ax2.set_title("Accuracy")
ax2.set_xlabel("epoch")
ax2.set_ylabel("accuracy")
ax2.set_ylim(0, 1)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 3종 task 비교 학습

In [ ]:
task_configs = [
    ("multiclass", cross_entropy,        cross_entropy_grad,        accuracy,        "Accuracy",  "steelblue"),
    ("binary",     binary_cross_entropy, binary_cross_entropy_grad, binary_accuracy, "Binary Acc","#E87B4C"),
    ("regression", mse,                  mse_grad,                  r2_score,        "R²",        "mediumseagreen"),
]

N_SIM, EPOCHS_SIM, LR_SIM = 256, 50, 0.05
rng_cmp = np.random.default_rng(42)
X_cmp   = rng_cmp.normal(0, 1, (N_SIM, 784)).astype(np.float32)
L_cmp   = rng_cmp.integers(0, 10, N_SIM).astype(np.uint8)

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle("MLP Training — 3 Tasks (synthetic)", fontsize=13, fontweight="bold")

for col, (task, loss_fn, grad_fn, metric_fn, metric_label, color) in enumerate(task_configs):
    T_cmp   = transform_targets(L_cmp, task)
    model_c = MLP(task=task, seed=0)

    losses, metrics = [], []
    for _ in range(EPOCHS_SIM):
        out = model_c.forward(X_cmp)
        losses.append(float(loss_fn(out, T_cmp)))
        metrics.append(float(metric_fn(out, T_cmp)))

        grad = grad_fn(out, T_cmp)
        model_c.backward(grad)
        for p, g in zip(model_c.params, model_c.grads):
            p -= LR_SIM * g

    axes[0, col].plot(losses, color=color, linewidth=2)
    axes[0, col].set_title(f"{task} — Loss")
    axes[0, col].set_xlabel("epoch")
    axes[0, col].set_ylabel("loss")
    axes[0, col].grid(True, alpha=0.3)

    axes[1, col].plot(metrics, color=color, linewidth=2)
    axes[1, col].set_title(f"{task} — {metric_label}")
    axes[1, col].set_xlabel("epoch")
    axes[1, col].set_ylabel(metric_label)
    axes[1, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. params in-place 업데이트 검증

In [ ]:
# params[0]과 model.net.layers[0].w가 같은 객체인지 확인
model_check = MLP(task="multiclass", seed=0)

first_layer_w = model_check.net.layers[0].w
w_before = first_layer_w.copy()

# 한 step 수행
x_chk = rng.normal(0, 1, (8, 784)).astype(np.float32)
lbls  = rng.integers(0, 10, 8).astype(np.uint8)
tgt   = transform_targets(lbls, "multiclass")

out_chk = model_check.forward(x_chk)
model_check.backward(cross_entropy_grad(out_chk, tgt))

# params[0]을 통해 in-place 업데이트
model_check.params[0] -= 0.01 * model_check.grads[0]

print("params[0] is net.layers[0].w :", model_check.params[0] is first_layer_w)
print("w 변경 여부 (업데이트 후)    :", not np.allclose(first_layer_w, w_before))

> **핵심**: `model.params`는 레이어 내부 `w`, `b` 배열과 같은 메모리를 참조한다.  
> `params[i] -= lr * grads[i]` 가 in-place(`-=`)이므로 실제 레이어 파라미터가 변경된다.  
> `params[i] = params[i] - lr * grads[i]` (새 객체 할당)은 효과가 없음에 주의한다.

## 7. 정리

```
MLP 구조:
  Linear(784→256) → Sigmoid → Linear(256→128) → Sigmoid → Linear(128→output_dim)
```

| task | output_dim | loss | metric |
|---|---|---|---|
| multiclass | 10 | cross_entropy | accuracy |
| binary | 1 | binary_cross_entropy | binary_accuracy |
| regression | 1 | mse | r2_score |

**수동 학습 루프 패턴**
```python
logits   = model.forward(x)
loss     = loss_fn(logits, targets)
grad_out = grad_fn(logits, targets)
model.backward(grad_out)
for p, g in zip(model.params, model.grads):
    p -= lr * g          # in-place 업데이트 필수
```

**핵심 설계 원칙**
- `forward()`는 raw logits를 반환한다. activation은 `losses.py`가 처리한다.
- 파라미터 업데이트는 반드시 in-place(`-=`)로 수행해야 레이어 내부 배열이 갱신된다.
- Stage 4에서는 이 수동 루프를 `Trainer`, `Evaluator`, `Optimizer`로 추상화한다.